In [35]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import datetime

pd.set_option('display.max_columns', None)
# pd.reset_option('display.max_columns')

loan_data = pd.read_csv('loan_data_2007_2014.csv', index_col=0, low_memory=False)

In [36]:
# Preprocessing raw data

loan_data['emp_length_int'] = loan_data['emp_length'].str.replace('+ years', '', regex=False)
loan_data['emp_length_int'] = loan_data['emp_length_int'].str.replace('< 1 year', '0')
loan_data['emp_length_int'] = loan_data['emp_length_int'].str.replace(' years', '')
loan_data['emp_length_int'] = loan_data['emp_length_int'].str.replace(' year', '')
loan_data['emp_length_int'] = loan_data['emp_length_int'].fillna('0')
loan_data['emp_length_int'] = loan_data['emp_length_int'].astype('uint8')

loan_data['term_int'] = loan_data['term'].str.replace(' months','')
loan_data['term_int'] = loan_data['term_int'].str.replace(' ','').astype('uint8')

loan_data['earliest_cr_line_date'] = pd.to_datetime(loan_data['earliest_cr_line'], format='%b-%y')
loan_data['mths_since_earliest_cr_line'] = np.floor((pd.to_datetime('2026-01-01') - loan_data['earliest_cr_line_date']).dt.days/30.44)
# loan_data['mths_since_earliest_cr_line'].describe()
loan_data['earliest_cr_line_date'] = np.where(loan_data['earliest_cr_line_date'] > pd.to_datetime('2026-01-01'), 
                                              loan_data['earliest_cr_line_date'] - pd.DateOffset(years=100), loan_data['earliest_cr_line_date'])
loan_data['mths_since_earliest_cr_line'] = np.floor((pd.to_datetime('2026-01-01') - loan_data['earliest_cr_line_date']).dt.days/30.44)
# loan_data['mths_since_earliest_cr_line'].describe()

loan_data['issue_d'] = pd.to_datetime(loan_data['issue_d'], format='%b-%y')
loan_data['mths_since_issue_d'] = np.floor((pd.to_datetime('2026-01-01') - loan_data['issue_d']).dt.days/30.44)
# loan_data['mths_since_issue_d'].describe()

# splitting variables into dummy variables, amending types to int8 instead of booleans
col_names = ['grade', 'sub_grade', 'home_ownership', 'verification_status', 'loan_status', 'purpose', 'addr_state', 'initial_list_status']
loan_data_dummies = []
for name in col_names:
    dummy = pd.get_dummies(loan_data[name], dtype='int8', prefix=name, prefix_sep=':')
    loan_data_dummies.append(dummy)

loan_data_dummies = pd.concat(loan_data_dummies, axis=1)
loan_data = pd.concat([loan_data, loan_data_dummies], axis=1)

# Checking null values
# null = loan_data.isnull().sum()
# null[null > 0]
loan_data['total_rev_hi_lim'] = loan_data['total_rev_hi_lim'].fillna(loan_data['funded_amnt'])

loan_data['annual_inc'] = loan_data['annual_inc'].fillna(loan_data['annual_inc'].mean())

cols = ['delinq_2yrs', 'inq_last_6mths', 'open_acc', 'pub_rec', 'total_acc', 'acc_now_delinq', 'emp_length_int', 'mths_since_earliest_cr_line']
loan_data[cols] = loan_data[cols].fillna(0)

In [37]:
# Dependent variable preparation

# loan_data['loan_status'].value_counts()
loan_data['good_bad'] = np.where(loan_data['loan_status'].isin(['Charged Off', 'Late (31-120 days)', 'Default', 'Does not meet the credit policy. Status:Charged Off']), 0, 1)

In [38]:
# Splitting data into test and train sets

# Stratify makes good/bad proportion in train and test sets to be the same
loan_data_inputs_train, loan_data_inputs_test, loan_data_targets_train, loan_data_targets_test = (
    train_test_split(loan_data.drop('good_bad', axis=1), loan_data['good_bad'], test_size=0.2, random_state=42, stratify=loan_data['good_bad']))

# print(loan_data_inputs_train.shape, loan_data_targets_train.shape, loan_data_inputs_test.shape, loan_data_targets_test.shape)

In [ ]:
# Assigning inputs and targets data
 
# Train
df_inputs = loan_data_inputs_train
df_targets = loan_data_targets_train

# Test
# df_inputs = loan_data_inputs_test
# df_targets = loan_data_targets_test

In [40]:
# Weight of Evidence (WoE) functions

# IV - information value
#        IV < 0.02 - no predictive power
# 0.02 < IV < 0.1  - weak
# 0.1  < IV < 0.3  - medium
# 0.3  < IV < 0.5  - strong
# 0.5  < IV        - suspiciously high

def woe_discrete(df_WoE, variable_name):
    df = pd.crosstab(df_WoE[variable_name], df_targets).rename(columns={0: 'bad', 1: 'good'}).rename_axis(None, axis=1)
    df['both'] = df['good'] + df['bad']
    df['good_proportion'] = df['good']/df['good'].sum()
    df['bad_proportion'] = df['bad']/df['bad'].sum()
    df['weight_of_evidence'] = np.log(df['good_proportion']/df['bad_proportion'])
    df['diff_woe'] = df['weight_of_evidence'].diff().abs()
    df['good_minus_bad'] = df['good_proportion'] - df['bad_proportion']
    df['information_value'] = df['good_minus_bad'] * df['weight_of_evidence']
    df['iv_sum'] = df['information_value'].sum()
    # sorting shows where is the highest default rate in given category
    return df.sort_values('weight_of_evidence').reset_index()


sns.set_theme()
def plot_by_woe(df_WoE):
    plt.figure(figsize=(12,4))
    plt.plot(range(len(df_WoE)), df_WoE['weight_of_evidence'], marker='o', linestyle='--', color='k')

    if len(df_WoE) > 20:
        for i, index in enumerate(df_WoE['weight_of_evidence']):
            plt.text(i, index + 0.2, str(i), fontsize=9, ha='center')

    plt.xlabel(df_WoE.columns[0])
    plt.ylabel('WoE')
    plt.title(f'Weight of Evidence by {df_WoE.columns[0]}')
    plt.xticks(range(len(df_WoE)), df_WoE.iloc[:, 0], rotation=270)
    plt.tight_layout()
    plt.show()


def woe_continuous(data_frame, variable_name):
    df = pd.concat([data_frame[variable_name], df_targets], axis = 1, join='inner')
    df = df.groupby(variable_name, observed=False)['good_bad'].value_counts()\
        .unstack().rename(columns={0:'bad', 1:'good'}).rename_axis(None, axis=1).reset_index()
    df['good'] = df['good'].fillna(0)
    df['bad'] = df['bad'].fillna(0)
    df['both'] = df['good'] + df['bad']
    df['good_proportion'] = df['good']/df['good'].sum()
    df['bad_proportion'] = df['bad']/df['bad'].sum()
    df['weight_of_evidence'] = np.log(df['good_proportion']/df['bad_proportion'])
    df['diff_woe'] = df['weight_of_evidence'].diff().abs()
    df['good_minus_bad'] = df['good_proportion'] - df['bad_proportion']
    df['information_value'] = df['good_minus_bad'] * df['weight_of_evidence']
    df['iv_sum'] = df['information_value'].sum()
    return df

In [41]:
# Discrete variables preperation

# groupped by low amount of data / similar WoE

# home_ownership
df_inputs['home_ownership:RENT_OTHER_NONE_ANY'] = sum([df_inputs['home_ownership:ANY'], df_inputs['home_ownership:NONE'],
                                                       df_inputs['home_ownership:OTHER'], df_inputs['home_ownership:RENT']])

# addr_state

df_inputs['addr_state:NE_IA_NV_FL_HI_AL'] = sum([df_inputs['addr_state:NE'], df_inputs['addr_state:IA'], df_inputs['addr_state:NV'], df_inputs['addr_state:FL'],
                                                 df_inputs['addr_state:HI'], df_inputs['addr_state:AL']])

df_inputs['addr_state:NM_VA'] = sum([df_inputs['addr_state:NM'], df_inputs['addr_state:VA']])

df_inputs['addr_state:OK_TN_MO_LA_MD_NC'] = sum([df_inputs['addr_state:OK'], df_inputs['addr_state:TN'], df_inputs['addr_state:MO'], df_inputs['addr_state:LA'],
                                                 df_inputs['addr_state:MD'], df_inputs['addr_state:NC']])

df_inputs['addr_state:UT_KY_AZ_NJ'] = sum([df_inputs['addr_state:UT'], df_inputs['addr_state:KY'], df_inputs['addr_state:AZ'], df_inputs['addr_state:NJ']])

df_inputs['addr_state:AR_MI_PA_OH_MN'] = sum([df_inputs['addr_state:AR'], df_inputs['addr_state:MI'], df_inputs['addr_state:PA'], df_inputs['addr_state:OH'],
                                              df_inputs['addr_state:MN']])

df_inputs['addr_state:RI_MA_DE_SD_IN'] = sum([df_inputs['addr_state:RI'], df_inputs['addr_state:MA'], df_inputs['addr_state:DE'], df_inputs['addr_state:SD'],
                                              df_inputs['addr_state:IN']])

df_inputs['addr_state:GA_WA_OR'] = sum([df_inputs['addr_state:GA'], df_inputs['addr_state:WA'], df_inputs['addr_state:OR']])

df_inputs['addr_state:WI_MT'] = sum([df_inputs['addr_state:WI'], df_inputs['addr_state:MT']])

df_inputs['addr_state:IL_CT'] = sum([df_inputs['addr_state:IL'], df_inputs['addr_state:CT']])

df_inputs['addr_state:KS_SC_CO_VT_AK_MS'] = sum([df_inputs['addr_state:KS'], df_inputs['addr_state:SC'], df_inputs['addr_state:CO'], df_inputs['addr_state:VT'],
                                                 df_inputs['addr_state:AK'], df_inputs['addr_state:MS']])

df_inputs['addr_state:WV_NH_WY_DC_ME_ID'] = sum([df_inputs['addr_state:WV'], df_inputs['addr_state:NH'], df_inputs['addr_state:WY'], df_inputs['addr_state:DC'],
                                                 df_inputs['addr_state:ME'], df_inputs['addr_state:ID']])

# purpose
df_inputs['purpose:educ__sm_b__wedd__ren_en__mov__house'] = sum([df_inputs['purpose:educational'], df_inputs['purpose:small_business'], df_inputs['purpose:wedding'], 
                                                                 df_inputs['purpose:renewable_energy'], df_inputs['purpose:moving'], df_inputs['purpose:house']])

df_inputs['purpose:oth__med__vacation'] = sum([df_inputs['purpose:other'], df_inputs['purpose:medical'], df_inputs['purpose:vacation']])

df_inputs['purpose:major_purch__car__home_impr'] = sum([df_inputs['purpose:major_purchase'], df_inputs['purpose:car'], df_inputs['purpose:home_improvement']])

# woe_discrete(df_inputs, 'abc')
# plot_by_woe(woe)

In [42]:
# Continuous variables preperation

# term_int (for how long the loan is taken) - WoE has easy to observe direction, even if iv_sum is small, data matters
df_inputs['term_int:36'] = np.where(df_inputs['term_int'] == 36, 1 ,0)
df_inputs['term_int:60'] = np.where(df_inputs['term_int'] == 60, 1 ,0)

# emp_length_int
df_inputs['emp_length_int:0'] = np.where(df_inputs['emp_length_int'].isin([0]), 1, 0)
df_inputs['emp_length_int:1'] = np.where(df_inputs['emp_length_int'].isin([1]), 1, 0)
df_inputs['emp_length_int:2-4'] = np.where(df_inputs['emp_length_int'].isin(range(2, 5)), 1, 0)
df_inputs['emp_length_int:5-6'] = np.where(df_inputs['emp_length_int'].isin(range(5, 7)), 1, 0)
df_inputs['emp_length_int:7-9'] = np.where(df_inputs['emp_length_int'].isin(range(7, 10)), 1, 0)
df_inputs['emp_length_int:10'] = np.where(df_inputs['emp_length_int'].isin([10]), 1, 0)

# mths_since_issue_d, need to do fine-classing as there are too many different outputs
# df_inputs['mths_since_issue_d_factor'] = pd.cut(df_inputs['mths_since_issue_d'], 50)
df_inputs['mths_since_issue_d:<135'] = np.where(df_inputs['mths_since_issue_d'].isin(range(135)), 1, 0)
df_inputs['mths_since_issue_d:135-136'] = np.where(df_inputs['mths_since_issue_d'].isin(range(135, 137)), 1, 0)
df_inputs['mths_since_issue_d:137-138'] = np.where(df_inputs['mths_since_issue_d'].isin(range(137, 139)), 1, 0)
df_inputs['mths_since_issue_d:139-145'] = np.where(df_inputs['mths_since_issue_d'].isin(range(139, 146)), 1, 0)
df_inputs['mths_since_issue_d:146-149'] = np.where(df_inputs['mths_since_issue_d'].isin(range(146, 150)), 1, 0)
df_inputs['mths_since_issue_d:150-161'] = np.where(df_inputs['mths_since_issue_d'].isin(range(150, 162)), 1, 0)
df_inputs['mths_since_issue_d:162-181'] = np.where(df_inputs['mths_since_issue_d'].isin(range(162, 182)), 1, 0)
df_inputs['mths_since_issue_d:>181'] = np.where(df_inputs['mths_since_issue_d'].isin(range(182, int(df_inputs['mths_since_issue_d'].max()) + 1)), 1, 0)

# int_rate
# df_inputs['int_rate_factor'] = pd.cut(df_inputs['int_rate'], 50)
df_inputs['int_rate:<9.548'] = np.where((df_inputs['int_rate'] <= 9.548), 1, 0)
df_inputs['int_rate:9.548-12.025'] = np.where((df_inputs['int_rate'] > 9.548) & (df_inputs['int_rate'] <= 12.025), 1, 0)
df_inputs['int_rate:12.025-15.74'] = np.where((df_inputs['int_rate'] > 12.025) & (df_inputs['int_rate'] <= 15.74), 1, 0)
df_inputs['int_rate:15.74-20.281'] = np.where((df_inputs['int_rate'] > 15.74) & (df_inputs['int_rate'] <= 20.281), 1, 0)
df_inputs['int_rate:>20.281'] = np.where((df_inputs['int_rate'] > 20.281), 1, 0)

# funded_amnt - weak iv_sum, woe with no specific direction
# df_inputs['funded_amnt_factor'] = pd.cut(df_inputs['funded_amnt'], 50)

# mths_since_earliest_cr_line (when person took first loan)
# df_inputs['mths_since_earliest_cr_line_factor'] = pd.cut(df_inputs['mths_since_earliest_cr_line'], 50)
df_inputs['mths_since_earliest_cr_line:<237'] = np.where(df_inputs['mths_since_earliest_cr_line'].isin(range(237)), 1, 0)
df_inputs['mths_since_earliest_cr_line:238-261'] = np.where(df_inputs['mths_since_earliest_cr_line'].isin(range(237, 262)), 1, 0)
df_inputs['mths_since_earliest_cr_line:262-344'] = np.where(df_inputs['mths_since_earliest_cr_line'].isin(range(262, 345)), 1, 0)
df_inputs['mths_since_earliest_cr_line:345-367'] = np.where(df_inputs['mths_since_earliest_cr_line'].isin(range(345, 368)), 1, 0)
df_inputs['mths_since_earliest_cr_line:368-449'] = np.where(df_inputs['mths_since_earliest_cr_line'].isin(range(368, 450)), 1, 0)
df_inputs['mths_since_earliest_cr_line:>449'] = np.where(df_inputs['mths_since_earliest_cr_line'].isin(range(450, int(df_inputs['mths_since_earliest_cr_line'].max()) + 1)), 1, 0)

# installment (monthly payment) - low iv and constant woe, term, int rate and funded amt covers this one
# df_inputs['installment_factor'] = pd.cut(df_inputs['installment'], 50)

# delinq_2yrs (zaleglosci w dniach w ostatnich 2 latach)
df_inputs['delinq_2yrs:0'] = np.where((df_inputs['delinq_2yrs'] == 0), 1, 0)
df_inputs['delinq_2yrs:1-3'] = np.where((df_inputs['delinq_2yrs'] >= 1) & (df_inputs['delinq_2yrs'] <= 3), 1, 0)
df_inputs['delinq_2yrs:>3'] = np.where((df_inputs['delinq_2yrs'] > 3), 1, 0)

# inq_last_6mths (loan requests within 6m)
df_inputs['inq_last_6mths:0'] = np.where((df_inputs['inq_last_6mths'] == 0), 1, 0)
df_inputs['inq_last_6mths:1-2'] = np.where((df_inputs['inq_last_6mths'] >= 1) & (df_inputs['inq_last_6mths'] <= 2), 1, 0)
df_inputs['inq_last_6mths:3-6'] = np.where((df_inputs['inq_last_6mths'] >= 3) & (df_inputs['inq_last_6mths'] <= 6), 1, 0)
df_inputs['inq_last_6mths:>6'] = np.where((df_inputs['inq_last_6mths'] > 6), 1, 0)

# open_acc (active liabilites)
df_inputs['open_acc:0'] = np.where((df_inputs['open_acc'] == 0), 1, 0)
df_inputs['open_acc:1-3'] = np.where((df_inputs['open_acc'] >= 1) & (df_inputs['open_acc'] <= 3), 1, 0)
df_inputs['open_acc:4-12'] = np.where((df_inputs['open_acc'] >= 4) & (df_inputs['open_acc'] <= 12), 1, 0)
df_inputs['open_acc:13-17'] = np.where((df_inputs['open_acc'] >= 13) & (df_inputs['open_acc'] <= 17), 1, 0)
df_inputs['open_acc:18-22'] = np.where((df_inputs['open_acc'] >= 18) & (df_inputs['open_acc'] <= 22), 1, 0)
df_inputs['open_acc:23-25'] = np.where((df_inputs['open_acc'] >= 23) & (df_inputs['open_acc'] <= 25), 1, 0)
df_inputs['open_acc:26-30'] = np.where((df_inputs['open_acc'] >= 26) & (df_inputs['open_acc'] <= 30), 1, 0)
df_inputs['open_acc:>30'] = np.where((df_inputs['open_acc'] > 30), 1, 0)

# pub_rec (defaults in the past etc)
df_inputs['pub_rec:0-2'] = np.where((df_inputs['pub_rec'] >= 0) & (df_inputs['pub_rec'] <= 2), 1, 0)
df_inputs['pub_rec:3-4'] = np.where((df_inputs['pub_rec'] >= 3) & (df_inputs['pub_rec'] <= 4), 1, 0)
df_inputs['pub_rec:>4'] = np.where((df_inputs['pub_rec'] > 4), 1, 0)

# total_acc (closed and active liabilities)
# df_inputs['total_acc_factor'] = pd.cut(df_inputs['total_acc'], 50)
df_inputs['total_acc:<=27'] = np.where((df_inputs['total_acc'] <= 27), 1, 0)
df_inputs['total_acc:28-51'] = np.where((df_inputs['total_acc'] >= 28) & (df_inputs['total_acc'] <= 51), 1, 0)
df_inputs['total_acc:>51'] = np.where((df_inputs['total_acc'] > 51), 1, 0)

# acc_now_delinq (number of liabilies where late payments are)
df_inputs['acc_now_delinq:0'] = np.where(df_inputs['acc_now_delinq'] == 0, 1, 0)
df_inputs['acc_now_delinq:>0'] = np.where(df_inputs['acc_now_delinq'] > 0, 1, 0)

# total_rev_hi_min
# df_inputs['total_rev_hi_lim_factor'] = pd.cut(df_inputs['total_rev_hi_lim'], 50)
df_inputs['total_rev_hi_lim:<=5K'] = np.where((df_inputs['total_rev_hi_lim'] <= 5000), 1, 0)
df_inputs['total_rev_hi_lim:5K-10K'] = np.where((df_inputs['total_rev_hi_lim'] > 5000) & (df_inputs['total_rev_hi_lim'] <= 10000), 1, 0)
df_inputs['total_rev_hi_lim:10K-20K'] = np.where((df_inputs['total_rev_hi_lim'] > 10000) & (df_inputs['total_rev_hi_lim'] <= 20000), 1, 0)
df_inputs['total_rev_hi_lim:20K-30K'] = np.where((df_inputs['total_rev_hi_lim'] > 20000) & (df_inputs['total_rev_hi_lim'] <= 30000), 1, 0)
df_inputs['total_rev_hi_lim:30K-40K'] = np.where((df_inputs['total_rev_hi_lim'] > 30000) & (df_inputs['total_rev_hi_lim'] <= 40000), 1, 0)
df_inputs['total_rev_hi_lim:40K-55K'] = np.where((df_inputs['total_rev_hi_lim'] > 40000) & (df_inputs['total_rev_hi_lim'] <= 55000), 1, 0)
df_inputs['total_rev_hi_lim:55K-95K'] = np.where((df_inputs['total_rev_hi_lim'] > 55000) & (df_inputs['total_rev_hi_lim'] <= 95000), 1, 0)
df_inputs['total_rev_hi_lim:>95K'] = np.where((df_inputs['total_rev_hi_lim'] > 95000), 1, 0)

# installment
# df_inputs['installment_factor'] = pd.cut(df_inputs['installment'], 50)

# Dane annual_inc
# df_temp = df_inputs.loc[df_inputs['annual_inc'] < 140_000, :].copy()
# df_temp['annual_inc_factor'] = pd.cut(df_temp['annual_inc'], 50)
df_inputs['annual_inc:<20K'] = np.where((df_inputs['annual_inc'] <= 20000), 1, 0)
df_inputs['annual_inc:20K-30K'] = np.where((df_inputs['annual_inc'] > 20000) & (df_inputs['annual_inc'] <= 30000), 1, 0)
df_inputs['annual_inc:30K-40K'] = np.where((df_inputs['annual_inc'] > 30000) & (df_inputs['annual_inc'] <= 40000), 1, 0)
df_inputs['annual_inc:40K-50K'] = np.where((df_inputs['annual_inc'] > 40000) & (df_inputs['annual_inc'] <= 50000), 1, 0)
df_inputs['annual_inc:50K-60K'] = np.where((df_inputs['annual_inc'] > 50000) & (df_inputs['annual_inc'] <= 60000), 1, 0)
df_inputs['annual_inc:60K-70K'] = np.where((df_inputs['annual_inc'] > 60000) & (df_inputs['annual_inc'] <= 70000), 1, 0)
df_inputs['annual_inc:70K-80K'] = np.where((df_inputs['annual_inc'] > 70000) & (df_inputs['annual_inc'] <= 80000), 1, 0)
df_inputs['annual_inc:80K-90K'] = np.where((df_inputs['annual_inc'] > 80000) & (df_inputs['annual_inc'] <= 90000), 1, 0)
df_inputs['annual_inc:90K-100K'] = np.where((df_inputs['annual_inc'] > 90000) & (df_inputs['annual_inc'] <= 100000), 1, 0)
df_inputs['annual_inc:100K-120K'] = np.where((df_inputs['annual_inc'] > 100000) & (df_inputs['annual_inc'] <= 120000), 1, 0)
df_inputs['annual_inc:120K-140K'] = np.where((df_inputs['annual_inc'] > 120000) & (df_inputs['annual_inc'] <= 140000), 1, 0)
df_inputs['annual_inc:>140K'] = np.where((df_inputs['annual_inc'] > 140000), 1, 0)

# months since last deliquency (woe formula by default ignores N/A/matches indices)
# df_temp = df_inputs[df_inputs['mths_since_last_delinq'].notnull()].copy()
# df_temp['mths_since_last_delinq_factor'] = pd.cut(df_temp['mths_since_last_delinq'], 50)
df_inputs['mths_since_last_delinq:N/A'] = np.where(df_inputs['mths_since_last_delinq'].isna(), 1, 0)
df_inputs['mths_since_last_delinq:0-3'] = np.where((df_inputs['mths_since_last_delinq'] >= 0) & (df_inputs['mths_since_last_delinq'] <= 3), 1, 0)
df_inputs['mths_since_last_delinq:4-30'] = np.where((df_inputs['mths_since_last_delinq'] >= 4) & (df_inputs['mths_since_last_delinq'] <= 30), 1, 0)
df_inputs['mths_since_last_delinq:31-56'] = np.where((df_inputs['mths_since_last_delinq'] >= 31) & (df_inputs['mths_since_last_delinq'] <= 56), 1, 0)
df_inputs['mths_since_last_delinq:>56'] = np.where((df_inputs['mths_since_last_delinq'] > 56), 1, 0)

# dti
# df_temp = df_inputs[df_inputs['dti'] < 35].copy()
# df_temp['dti_factor'] = pd.cut(df_temp['dti'], 50)
df_inputs['dti:<=1.4'] = np.where((df_inputs['dti'] <= 1.4), 1, 0)
df_inputs['dti:1.4-3.5'] = np.where((df_inputs['dti'] > 1.4) & (df_inputs['dti'] <= 3.5), 1, 0)
df_inputs['dti:3.5-7.7'] = np.where((df_inputs['dti'] > 3.5) & (df_inputs['dti'] <= 7.7), 1, 0)
df_inputs['dti:7.7-10.5'] = np.where((df_inputs['dti'] > 7.7) & (df_inputs['dti'] <= 10.5), 1, 0)
df_inputs['dti:10.5-16.1'] = np.where((df_inputs['dti'] > 10.5) & (df_inputs['dti'] <= 16.1), 1, 0)
df_inputs['dti:16.1-20.3'] = np.where((df_inputs['dti'] > 16.1) & (df_inputs['dti'] <= 20.3), 1, 0)
df_inputs['dti:20.3-21.7'] = np.where((df_inputs['dti'] > 20.3) & (df_inputs['dti'] <= 21.7), 1, 0)
df_inputs['dti:21.7-22.4'] = np.where((df_inputs['dti'] > 21.7) & (df_inputs['dti'] <= 22.4), 1, 0)
df_inputs['dti:22.4-35'] = np.where((df_inputs['dti'] > 22.4) & (df_inputs['dti'] <= 35), 1, 0)
df_inputs['dti:>35'] = np.where((df_inputs['dti'] > 35), 1, 0)

# mths_since_last_record
# df_inputs['mths_since_last_record_factor'] = pd.cut(df_inputs['mths_since_last_record'], 40)
df_inputs['mths_since_last_record:N/A'] = np.where((df_inputs['mths_since_last_record'].isna()), 1, 0)
df_inputs['mths_since_last_record:0-2'] = np.where((df_inputs['mths_since_last_record'] >= 0) & (df_inputs['mths_since_last_record'] <= 2), 1, 0)
df_inputs['mths_since_last_record:3-20'] = np.where((df_inputs['mths_since_last_record'] >= 3) & (df_inputs['mths_since_last_record'] <= 20), 1, 0)
df_inputs['mths_since_last_record:21-31'] = np.where((df_inputs['mths_since_last_record'] >= 21) & (df_inputs['mths_since_last_record'] <= 31), 1, 0)
df_inputs['mths_since_last_record:32-80'] = np.where((df_inputs['mths_since_last_record'] >= 32) & (df_inputs['mths_since_last_record'] <= 80), 1, 0)
df_inputs['mths_since_last_record:81-86'] = np.where((df_inputs['mths_since_last_record'] >= 81) & (df_inputs['mths_since_last_record'] <= 86), 1, 0)
df_inputs['mths_since_last_record:>86'] = np.where((df_inputs['mths_since_last_record'] > 86), 1, 0)


# woe_continuous(df_inputs, 'abc')
# plot_by_woe(woe_continuous(df_inputs, 'abc'))

C:\Users\Mateusz\AppData\Local\Temp\ipykernel_15884\797595407.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_inputs['int_rate:15.74-20.281'] = np.where((df_inputs['int_rate'] > 15.74) & (df_inputs['int_rate'] <= 20.281), 1, 0)
C:\Users\Mateusz\AppData\Local\Temp\ipykernel_15884\797595407.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_inputs['int_rate:>20.281'] = np.where((df_inputs['int_rate'] > 20.281), 1, 0)
C:\Users\Mateusz\AppData\Local\Temp\ipykernel_15884\797595407.py:39: PerformanceWarning: DataFrame is 

In [ ]:
# Overwritten train and test data

# TRAIN
loan_data_inputs_train = df_inputs

# TEST
# loan_data_inputs_test = df_inputs

In [ ]:
# Saving Data

loan_data_inputs_train.to_csv('loan_data_inputs_train.csv')
loan_data_targets_train.to_csv('loan_data_targets_train.csv')
# loan_data_inputs_test.to_csv('loan_data_inputs_test.csv')
# loan_data_targets_test.to_csv('loan_data_targets_test.csv')